In [ ]:
# Kp index from NOAA

import requests

url = "https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json"
data = requests.get(url).json()

latest_kp = float(data[-1][1])
print("Kp:", latest_kp)


In [ ]:
# Solar wind data and speed data from NOAA

url = "https://services.swpc.noaa.gov/products/solar-wind/plasma-1-day.json"
data = requests.get(url).json()

latest_speed = float(data[-1][2])  # km/s
density = float(data[-1][1])       # particles/cm^3

print("Speed:", latest_speed, "Density:", density)


In [ ]:
# IMF Bz data from NOAA

url = "https://services.swpc.noaa.gov/products/solar-wind/mag-1-day.json"
data = requests.get(url).json()

bz = float(data[-1][6])  # nT
print("IMF Bz:", bz)


In [ ]:
url = "https://services.swpc.noaa.gov/json/ovation_aurora_latest.json"
data = requests.get(url).json()

# Example access
lat = data["coordinates"][0]
lon = data["coordinates"][1]
prob = data["probability"]

print("Sample probability:", prob[0])


In [ ]:

import requests

headers = {"User-Agent": "aurora-project"}
url = "https://api.weather.gov/points/64.84,-147.72"  # Fairbanks, AK

data = requests.get(url, headers=headers).json()
forecast_url = data["properties"]["forecast"]

forecast = requests.get(forecast_url, headers=headers).json()
print(forecast["properties"]["periods"][0]["shortForecast"])



In [ ]:
import requests
from datetime import datetime

KP_URL = "https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json"
PLASMA_URL = "https://services.swpc.noaa.gov/products/solar-wind/plasma-1-day.json"
MAG_URL = "https://services.swpc.noaa.gov/products/solar-wind/mag-1-day.json"
OVATION_URL = "https://services.swpc.noaa.gov/json/ovation_aurora_latest.json"
ALERTS_URL = "https://services.swpc.noaa.gov/products/alerts.json"


def fetch_kp():
    data = requests.get(KP_URL, timeout=10).json()
    timestamp, kp = data[-1][0], float(data[-1][1])
    return kp, timestamp


def fetch_solar_wind():
    data = requests.get(PLASMA_URL, timeout=10).json()
    latest = data[-1]
    return {
        "speed": float(latest[2]),      # km/s
        "density": float(latest[1]),    # particles / cm^3
        "time": latest[0]
    }


def fetch_imf():
    data = requests.get(MAG_URL, timeout=10).json()
    latest = data[-1]
    return {
        "bz": float(latest[6]),         # nT
        "bt": float(latest[1]),         # total field strength
        "time": latest[0]
    }


def fetch_aurora_oval():
    data = requests.get(OVATION_URL, timeout=10).json()

    aurora_values = []

    for feature in data.get("features", []):
        props = feature.get("properties", {})
        if "aurora" in props:
            aurora_values.append(props["aurora"])

    if aurora_values:
        return {
            "max_probability": max(aurora_values),
            "avg_probability": sum(aurora_values) / len(aurora_values),
            "model_time": data.get("Forecast Time", "unknown")
        }
    else:
        return {
            "max_probability": None,
            "avg_probability": None,
            "model_time": "no data"
        }


def fetch_alerts():
    data = requests.get(ALERTS_URL, timeout=10).json()
    return [alert.get("message", "") for alert in data]


def fetch_all_space_weather():
    print("\nAURORA SPACE WEATHER")
    print("=" * 45)

    kp, kp_time = fetch_kp()
    solar = fetch_solar_wind()
    imf = fetch_imf()
    oval = fetch_aurora_oval()
    alerts = fetch_alerts()

    print(f"Time: {datetime.utcnow().strftime('%Y-%m-%d %H:%M')}")

    print("\nGeomagnetic Conditions")
    print(f"  • Kp Index: {kp}  (at {kp_time})")

    print("\nSolar Wind")
    print(f"  • Speed: {solar['speed']:.0f} km/s")
    print(f"  • Density: {solar['density']:.1f} p/cm³")

    print("\nInterplanetary Magnetic Field")
    print(f"  • Bz: {imf['bz']:.1f} nT")
    print(f"  • Bt: {imf['bt']:.1f} nT")

    print("\nAurora Oval (OVATION Model)")
    if oval["max_probability"] is not None:
        print(f"  • Max probability: {oval['max_probability']:.1f}%")
        print(f"  • Avg probability: {oval['avg_probability']:.1f}%")
        print(f"  • Model time: {oval['model_time']}")
    else:
        print("  • No aurora probability data available")

    print("\nActive Space Weather Alerts")
    if alerts:
        for alert in alerts:
            print(f"  • {alert}")
    else:
        print("  • None")

    print("\n✔ Data fetch complete\n")


if __name__ == "__main__":
    fetch_all_space_weather()



AURORA SPACE WEATHER
Time: 2026-02-13 02:30

Geomagnetic Conditions
  • Kp Index: 1.67  (at 2026-02-12 21:00:00.000)

Solar Wind
  • Speed: 386 km/s
  • Density: 0.3 p/cm³

Interplanetary Magnetic Field
  • Bz: 7.5 nT
  • Bt: 1.6 nT

Aurora Oval (OVATION Model)
  • No aurora probability data available

Active Space Weather Alerts
  • Space Weather Message Code: ALTK04
Serial Number: 2631
Issue Time: 2026 Feb 13 0213 UTC

ALERT: Geomagnetic K-index of 4
 Threshold Reached: 2026 Feb 13 0213 UTC
Synoptic Period: 0000-0300 UTC
 
Active Warning: Yes

NOAA Space Weather Scale descriptions can be found at
www.swpc.noaa.gov/noaa-scales-explanation

Potential Impacts: Area of impact primarily poleward of 65 degrees Geomagnetic Latitude.
Induced Currents - Weak power grid fluctuations can occur.
Aurora - Aurora may be visible at high latitudes such as Canada and Alaska.
  • Space Weather Message Code: WARK04
Serial Number: 5253
Issue Time: 2026 Feb 13 0211 UTC

Valid From: 2026 Feb 13 0210 UTC
